# Module 3 + Module 4 + Module 5 Combined Notebook

This notebook runs the full pipeline in order: Module 3 preprocessing, Module 4 hybrid forecasting, and Module 5 price optimization. The code is kept simple, runnable, and production-ready, with explanations added where needed.

## 1) Mount Google Drive
We use Google Drive for permanent storage of inputs and outputs, so the files survive Colab runtime resets.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2) Import libraries
These libraries are used across the three modules for preprocessing, forecasting, optimization, and saving results.

In [2]:
import io
import os
import json
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from xgboost import XGBRegressor
from google.colab import files

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

try:
    from prophet import Prophet
except Exception:
    from fbprophet import Prophet

## 3) Upload CSV
Upload the retail sales CSV that will be processed by the pipeline.

In [3]:
uploaded = files.upload()
input_csv_name = next(iter(uploaded.keys()))
print('Uploaded file:', input_csv_name)

Saving indian_sme_retail_dataset.csv to indian_sme_retail_dataset.csv
Uploaded file: indian_sme_retail_dataset.csv


## 4) Configuration
This section stores the important paths and column names in one place.

In [4]:
@dataclass
class Module3Config:
    input_csv_path: str = input_csv_name
    output_dir: str = '/content/drive/MyDrive/module3_outputs'
    date_col: str = 'order_date'
    product_col: str = 'product_id'
    target_col: str = 'quantity_sold'
    price_col: str = 'unit_price_inr'
    sales_col: str = 'total_sales_inr'
    cost_col: str = 'cost_price_inr'
    profit_col: str = 'profit_inr'
    discount_col: str = 'discount_percent'
    lags: Tuple[int, ...] = (1, 7, 14)
    rolling_windows: Tuple[int, ...] = (3, 7, 14)
    fill_unknown_categoricals: bool = True
    minmax_scale_for_pricing: bool = True
    minmax_scale_for_anomaly: bool = True
    categorical_fill_value: str = 'Unknown'
    forecast_horizons: Tuple[int, ...] = (7, 30)

@dataclass
class Module4Config:
    forecasting_csv_path: str = '/content/drive/MyDrive/module3_outputs/module3_forecasting_dataset.csv'
    prophet_csv_path: str = '/content/drive/MyDrive/module3_outputs/module3_prophet_dataset.csv'
    output_dir: str = '/content/drive/MyDrive/module4_outputs'
    product_col: str = 'product_id'
    date_col: str = 'order_date'
    target_col: str = 'quantity_sold'
    test_size: float = 0.2
    random_state: int = 42
    horizons: Tuple[int, int] = (7, 30)
    min_history_points: int = 20

@dataclass
class Module5Config:
    pricing_csv_path: str = '/content/drive/MyDrive/module3_outputs/module3_pricing_dataset.csv'
    output_dir: str = '/content/drive/MyDrive/module5_outputs'
    product_col: str = 'product_id'
    date_col: str = 'order_date'
    target_col: str = 'quantity_sold'
    price_col: str = 'unit_price_inr'
    cost_col: str = 'cost_price_inr'
    sales_col: str = 'total_sales_inr'
    profit_col: str = 'profit_inr'
    test_size: float = 0.2
    random_state: int = 42
    price_grid_points: int = 40
    candidate_price_range: Tuple[float, float] = (0.7, 1.3)

config3 = Module3Config()
config4 = Module4Config()
config5 = Module5Config()
os.makedirs(config3.output_dir, exist_ok=True)
os.makedirs(config4.output_dir, exist_ok=True)
os.makedirs(config5.output_dir, exist_ok=True)
print('Module 3 output dir:', config3.output_dir)
print('Module 4 output dir:', config4.output_dir)
print('Module 5 output dir:', config5.output_dir)

Module 3 output dir: /content/drive/MyDrive/module3_outputs
Module 4 output dir: /content/drive/MyDrive/module4_outputs
Module 5 output dir: /content/drive/MyDrive/module5_outputs


## 5) Module 3 preprocessing class
This class cleans the data, removes duplicates, handles missing values, and creates features for forecasting, pricing, and anomaly detection.

In [5]:
class Module3Preprocessor:
    def __init__(self, config: Module3Config):
        self.config = config
        self.pricing_scaler = MinMaxScaler()
        self.anomaly_scaler = MinMaxScaler()
        self.label_encoders: Dict[str, LabelEncoder] = {}

    def load_data(self) -> pd.DataFrame:
        if not os.path.exists(self.config.input_csv_path):
            raise FileNotFoundError(f'Input CSV not found: {self.config.input_csv_path}')
        return pd.read_csv(self.config.input_csv_path)

    def standardize_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.columns = (
            df.columns.astype(str)
            .str.strip()
            .str.lower()
            .str.replace(' ', '_', regex=False)
            .str.replace('-', '_', regex=False)
        )
        return df

    def validate(self, df: pd.DataFrame):
        required = {self.config.date_col, self.config.product_col, self.config.target_col, self.config.price_col, self.config.sales_col}
        missing = sorted(list(required - set(df.columns)))
        if missing:
            raise ValueError(f'Missing required columns: {missing}')

    def convert_types(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df[self.config.date_col] = pd.to_datetime(df[self.config.date_col], errors='coerce')
        for col in [self.config.target_col, self.config.price_col, self.config.sales_col, self.config.cost_col, self.config.profit_col, self.config.discount_col]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        return df

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy().drop_duplicates()
        subset = [c for c in [self.config.date_col, self.config.product_col, self.config.target_col, self.config.price_col] if c in df.columns]
        df = df.dropna(subset=subset)
        if self.config.target_col in df.columns:
            df = df[df[self.config.target_col] >= 0]
        if self.config.price_col in df.columns:
            df = df[df[self.config.price_col] >= 0]
        if self.config.sales_col in df.columns:
            df = df[df[self.config.sales_col] >= 0]
        return df.reset_index(drop=True)

    def fill_missing(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
        if self.config.product_col in df.columns:
            for col in numeric_cols:
                df[col] = df.groupby(self.config.product_col)[col].transform(lambda s: s.ffill().bfill())
        for col in numeric_cols:
            if df[col].isna().sum() > 0:
                df[col] = df[col].fillna(df[col].median())
        if self.config.fill_unknown_categoricals:
            for col in categorical_cols:
                df[col] = df[col].fillna(self.config.categorical_fill_value)
        return df

    def sort_data(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.sort_values([c for c in [self.config.product_col, self.config.date_col] if c in df.columns]).reset_index(drop=True)

    def create_time_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        d = df[self.config.date_col]
        df['day'] = d.dt.day
        df['day_of_week'] = d.dt.dayofweek
        df['week_of_year'] = d.dt.isocalendar().week.astype(int)
        df['month'] = d.dt.month
        df['quarter'] = d.dt.quarter
        df['year'] = d.dt.year
        df['is_weekend'] = d.dt.dayofweek.isin([5, 6]).astype(int)
        df['is_month_start'] = d.dt.is_month_start.astype(int)
        df['is_month_end'] = d.dt.is_month_end.astype(int)
        df['is_quarter_start'] = d.dt.is_quarter_start.astype(int)
        df['is_quarter_end'] = d.dt.is_quarter_end.astype(int)
        return df

    def create_business_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        if self.config.price_col in df.columns and self.config.cost_col in df.columns:
            df['unit_margin_inr'] = df[self.config.price_col] - df[self.config.cost_col]
        if self.config.sales_col in df.columns and self.config.target_col in df.columns:
            qty = df[self.config.target_col].replace(0, np.nan)
            df['avg_realized_price_inr'] = (df[self.config.sales_col] / qty).replace([np.inf, -np.inf], np.nan)
        if self.config.discount_col in df.columns:
            df['discount_flag'] = (df[self.config.discount_col] > 0).astype(int)
        if self.config.sales_col in df.columns:
            df['log_total_sales'] = np.log1p(df[self.config.sales_col])
        if self.config.target_col in df.columns:
            df['log_quantity_sold'] = np.log1p(df[self.config.target_col])
        return df

    def aggregate_daily(self, df: pd.DataFrame) -> pd.DataFrame:
        agg_map = {self.config.target_col: 'sum', self.config.sales_col: 'sum'}
        for col in [self.config.price_col, self.config.cost_col, self.config.discount_col]:
            if col in df.columns:
                agg_map[col] = 'mean'
        if self.config.profit_col in df.columns:
            agg_map[self.config.profit_col] = 'sum'

        def safe_mode(series):
            m = series.mode(dropna=True)
            return m.iloc[0] if len(m) else np.nan

        optional_mode_cols = ['product_name', 'category', 'sub_category', 'promotion_type', 'festival_season', 'stock_availability', 'weather_impact', 'demand_level', 'sales_channel']
        grouped = df.groupby([self.config.product_col, self.config.date_col], as_index=False).agg({**agg_map, **{c: safe_mode for c in optional_mode_cols if c in df.columns}})
        return grouped.sort_values([self.config.product_col, self.config.date_col]).reset_index(drop=True)

    def create_lags(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        for lag in self.config.lags:
            df[f'lag_{lag}'] = df.groupby(self.config.product_col)[self.config.target_col].shift(lag)
        return df

    def create_rollings(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        for window in self.config.rolling_windows:
            g = df.groupby(self.config.product_col)[self.config.target_col]
            df[f'rolling_mean_{window}'] = g.transform(lambda s: s.shift(1).rolling(window).mean())
            df[f'rolling_std_{window}'] = g.transform(lambda s: s.shift(1).rolling(window).std())
            df[f'rolling_min_{window}'] = g.transform(lambda s: s.shift(1).rolling(window).min())
            df[f'rolling_max_{window}'] = g.transform(lambda s: s.shift(1).rolling(window).max())
        return df

    def create_change_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df['qty_diff_1'] = df.groupby(self.config.product_col)[self.config.target_col].diff(1)
        df['qty_pct_change_1'] = df.groupby(self.config.product_col)[self.config.target_col].pct_change(1)
        if self.config.price_col in df.columns:
            df['price_diff_1'] = df.groupby(self.config.product_col)[self.config.price_col].diff(1)
            df['price_pct_change_1'] = df.groupby(self.config.product_col)[self.config.price_col].pct_change(1)
        return df

    def build_prophet_dataset(self, df_daily: pd.DataFrame) -> pd.DataFrame:
        prophet_df = df_daily[[self.config.product_col, self.config.date_col, self.config.target_col]].copy()
        return prophet_df.rename(columns={self.config.date_col: 'ds', self.config.target_col: 'y'}).sort_values([self.config.product_col, 'ds']).reset_index(drop=True)

    def encode_categoricals(self, df: pd.DataFrame, exclude_cols: Optional[list] = None) -> pd.DataFrame:
        df = df.copy()
        exclude_cols = exclude_cols or []
        cat_cols = [c for c in df.select_dtypes(include=['object']).columns.tolist() if c not in exclude_cols]
        for col in cat_cols:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            self.label_encoders[col] = le
        return df

    def build_pricing_dataset(self, df: pd.DataFrame) -> pd.DataFrame:
        pricing_df = self.create_time_features(df.copy())
        pricing_df = self.create_business_features(pricing_df).replace([np.inf, -np.inf], np.nan)
        pricing_df = pricing_df.dropna(subset=[self.config.target_col, self.config.price_col, self.config.sales_col]).reset_index(drop=True)
        pricing_df = self.encode_categoricals(pricing_df, exclude_cols=[self.config.date_col])
        numeric_cols = [c for c in pricing_df.select_dtypes(include=[np.number]).columns.tolist() if c != self.config.target_col]
        if numeric_cols:
            pricing_df[numeric_cols] = self.pricing_scaler.fit_transform(pricing_df[numeric_cols])
        return pricing_df

    def build_anomaly_dataset(self, df: pd.DataFrame) -> pd.DataFrame:
        anomaly_df = self.create_time_features(df.copy())
        anomaly_df = self.create_business_features(anomaly_df).replace([np.inf, -np.inf], np.nan)
        anomaly_df = anomaly_df.dropna().reset_index(drop=True)
        anomaly_df = self.encode_categoricals(anomaly_df, exclude_cols=[self.config.date_col])
        numeric_cols = anomaly_df.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_cols:
            anomaly_df[numeric_cols] = self.anomaly_scaler.fit_transform(anomaly_df[numeric_cols])
        return anomaly_df

    def finalize_forecasting_dataset(self, df_daily: pd.DataFrame) -> pd.DataFrame:
        df = self.create_time_features(df_daily.copy())
        df = self.create_business_features(df)
        df = self.create_lags(df)
        df = self.create_rollings(df)
        df = self.create_change_features(df)
        return df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    def build_summary(self, raw_df, cleaned_df, daily_df, forecast_df, pricing_df, anomaly_df):
        return pd.DataFrame({
            'dataset': ['raw_input', 'cleaned_transaction_data', 'daily_product_aggregated_data', 'forecasting_dataset', 'pricing_dataset', 'anomaly_dataset'],
            'rows': [len(raw_df), len(cleaned_df), len(daily_df), len(forecast_df), len(pricing_df), len(anomaly_df)],
            'columns': [raw_df.shape[1], cleaned_df.shape[1], daily_df.shape[1], forecast_df.shape[1], pricing_df.shape[1], anomaly_df.shape[1]]
        })

    def run(self) -> Dict[str, pd.DataFrame]:
        raw_df = self.standardize_columns(self.load_data())
        self.validate(raw_df)
        cleaned_df = self.sort_data(self.fill_missing(self.clean(self.convert_types(raw_df))))
        daily_df = self.aggregate_daily(cleaned_df)
        forecast_df = self.finalize_forecasting_dataset(daily_df)
        prophet_df = self.build_prophet_dataset(daily_df)
        pricing_df = self.build_pricing_dataset(cleaned_df)
        anomaly_df = self.build_anomaly_dataset(cleaned_df)
        summary_df = self.build_summary(raw_df, cleaned_df, daily_df, forecast_df, pricing_df, anomaly_df)

        os.makedirs(self.config.output_dir, exist_ok=True)
        cleaned_df.to_csv(os.path.join(self.config.output_dir, 'module3_cleaned_transactions.csv'), index=False)
        daily_df.to_csv(os.path.join(self.config.output_dir, 'module3_daily_product_data.csv'), index=False)
        forecast_df.to_csv(os.path.join(self.config.output_dir, 'module3_forecasting_dataset.csv'), index=False)
        prophet_df.to_csv(os.path.join(self.config.output_dir, 'module3_prophet_dataset.csv'), index=False)
        pricing_df.to_csv(os.path.join(self.config.output_dir, 'module3_pricing_dataset.csv'), index=False)
        anomaly_df.to_csv(os.path.join(self.config.output_dir, 'module3_anomaly_dataset.csv'), index=False)
        summary_df.to_csv(os.path.join(self.config.output_dir, 'module3_summary.csv'), index=False)
        with open(os.path.join(self.config.output_dir, 'module3_metadata.json'), 'w') as f:
            json.dump({'input_csv_path': self.config.input_csv_path, 'output_dir': self.config.output_dir}, f, indent=2)

        return {'raw_df': raw_df, 'cleaned_df': cleaned_df, 'daily_df': daily_df, 'forecast_df': forecast_df, 'prophet_df': prophet_df, 'pricing_df': pricing_df, 'anomaly_df': anomaly_df, 'summary_df': summary_df}

## 6) Run Module 3
This creates the cleaned datasets and saves them to Google Drive.

In [6]:
m3_processor = Module3Preprocessor(config3)
m3 = m3_processor.run()
display(m3['summary_df'])

,dataset,rows,columns
0,raw_input,10100,28
1,cleaned_transaction_data,9812,28
2,daily_product_aggregated_data,3387,17
3,forecasting_dataset,2868,52
4,pricing_dataset,9812,44
5,anomaly_dataset,9812,44


## 7) Module 4 hybrid forecasting class
This class trains the hybrid Prophet + XGBoost forecasting model and produces 7-day and 30-day forecasts.

In [7]:
class HybridDemandForecaster:
    def __init__(self, config: Module4Config):
        self.config = config
        self.prophet_models: Dict[str, Prophet] = {}
        self.xgb_model: Optional[XGBRegressor] = None
        self.feature_columns: List[str] = []
        self.metrics: Dict[str, float] = {}
        self.forecast_results: Dict[str, pd.DataFrame] = {}

    def prepare_base(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.sort_values([self.config.product_col, self.config.date_col]).reset_index(drop=True)

    def split_train_test(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        cutoff = int(len(df) * (1 - self.config.test_size))
        return df.iloc[:cutoff].copy(), df.iloc[cutoff:].copy()

    def xgb_feature_columns(self, df: pd.DataFrame) -> List[str]:
        exclude = {self.config.product_col, self.config.date_col, self.config.target_col, 'ds', 'y'}
        exclude.update([c for c in df.columns if c.startswith('order_') or c.endswith('_name')])
        return [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

    def fit_prophet_per_product(self, prophet_df: pd.DataFrame):
        models = {}
        for pid, group in prophet_df.groupby(self.config.product_col):
            g = group[['ds', 'y']].copy()
            if len(g) < self.config.min_history_points:
                continue
            n_changepoints = min(25, max(1, len(g) - 1))
            # Fix 7: multiplicative seasonality scales with trend (better for retail)
            m = Prophet(growth='linear', yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, seasonality_mode='multiplicative', interval_width=0.95, changepoint_prior_scale=0.05, n_changepoints=n_changepoints)
            m.fit(g)
            models[pid] = m
        self.prophet_models = models
        return models

    def prophet_in_sample(self, prophet_df: pd.DataFrame) -> pd.DataFrame:
        frames = []
        for pid, group in prophet_df.groupby(self.config.product_col):
            if pid not in self.prophet_models:
                continue
            m = self.prophet_models[pid]
            pred = m.predict(group[['ds']])[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
            pred[self.config.product_col] = pid
            pred = pred.merge(group[['ds', 'y']], on='ds', how='left')
            pred['residual'] = pred['y'] - pred['yhat']
            frames.append(pred)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=[self.config.product_col, 'ds', 'y', 'yhat', 'yhat_lower', 'yhat_upper', 'residual'])

    def train_xgb_residual_model(self, train_df: pd.DataFrame, residuals_df: pd.DataFrame):
        merged = train_df.merge(residuals_df[[self.config.product_col, 'ds', 'residual']], left_on=[self.config.product_col, self.config.date_col], right_on=[self.config.product_col, 'ds'], how='left')
        merged['residual'] = merged['residual'].fillna(0)
        self.feature_columns = self.xgb_feature_columns(merged)
        X = merged[self.feature_columns].copy()
        y = merged['residual']
        model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.1, reg_lambda=1.0, random_state=self.config.random_state)
        model.fit(X, y)
        self.xgb_model = model
        return model

    def evaluate(self, test_df: pd.DataFrame, residuals_df: pd.DataFrame) -> Dict[str, float]:
        merged = test_df.merge(residuals_df[[self.config.product_col, 'ds', 'yhat']], left_on=[self.config.product_col, self.config.date_col], right_on=[self.config.product_col, 'ds'], how='left')
        merged['yhat'] = merged['yhat'].ffill().fillna(merged[self.config.target_col].mean())
        for col in self.feature_columns:
            if col not in merged.columns:
                merged[col] = 0
        feature_cols = [c for c in self.feature_columns if c in merged.columns]
        X_test = merged[feature_cols].copy()
        pred_resid = self.xgb_model.predict(X_test)
        final_pred = merged['yhat'] + pred_resid
        y_true = merged[self.config.target_col].values
        mae = mean_absolute_error(y_true, final_pred)
        rmse = np.sqrt(mean_squared_error(y_true, final_pred))
        mape = np.mean(np.abs((y_true - final_pred) / np.maximum(np.abs(y_true), 1e-6))) * 100
        # Fix 8: sMAPE handles zero-demand products correctly (MAPE explodes when actual=0)
        _smape_denom = (np.abs(y_true) + np.abs(final_pred.values)) / 2
        smape = float(np.mean(np.where(_smape_denom == 0, 0, np.abs(y_true - final_pred.values) / _smape_denom)) * 100)
        self.metrics = {'MAE': float(mae), 'RMSE': float(rmse), 'MAPE': float(mape), 'sMAPE': smape}
        return self.metrics

    def forecast_horizon_for_product(self, product_df: pd.DataFrame, horizon: int) -> pd.DataFrame:
        pid = product_df[self.config.product_col].iloc[0]
        if pid not in self.prophet_models:
            return pd.DataFrame()
        m = self.prophet_models[pid]
        future = m.make_future_dataframe(periods=horizon, freq='D')
        prophet_fcst = m.predict(future)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
        prophet_fcst[self.config.product_col] = pid
        last_row = product_df.iloc[-1:].copy()
        future_rows = []
        for i in range(horizon):
            row = last_row.copy()
            row[self.config.date_col] = prophet_fcst['ds'].iloc[len(product_df) + i]
            row['day'] = row[self.config.date_col].dt.day.values[0]
            row['day_of_week'] = row[self.config.date_col].dt.dayofweek.values[0]
            row['week_of_year'] = row[self.config.date_col].dt.isocalendar().week.astype(int).values[0]
            row['month'] = row[self.config.date_col].dt.month.values[0]
            row['quarter'] = row[self.config.date_col].dt.quarter.values[0]
            row['year'] = row[self.config.date_col].dt.year.values[0]
            row['is_weekend'] = int(row[self.config.date_col].dt.dayofweek.values[0] in [5, 6])
            for lag in [1, 7, 14]:
                col = f'lag_{lag}'
                if col in row.columns:
                    row[col] = product_df[self.config.target_col].iloc[-lag] if len(product_df) >= lag else product_df[self.config.target_col].iloc[-1]
            for w in [3, 7, 14]:
                if f'rolling_mean_{w}' in row.columns:
                    vals = product_df[self.config.target_col].tail(w)
                    row[f'rolling_mean_{w}'] = float(vals.mean()) if len(vals) > 0 else float(product_df[self.config.target_col].iloc[-1])
                    row[f'rolling_std_{w}'] = float(vals.std()) if len(vals) > 1 else 0.0
                    row[f'rolling_min_{w}'] = float(vals.min()) if len(vals) > 0 else float(product_df[self.config.target_col].iloc[-1])
                    row[f'rolling_max_{w}'] = float(vals.max()) if len(vals) > 0 else float(product_df[self.config.target_col].iloc[-1])
            future_rows.append(row)
        future_df = pd.concat(future_rows, ignore_index=True)
        for col in self.feature_columns:
            if col not in future_df.columns:
                future_df[col] = 0
        pred_resid = self.xgb_model.predict(future_df[self.feature_columns])
        final = prophet_fcst.tail(horizon).copy()
        final['xgb_residual'] = pred_resid
        final['hybrid_yhat'] = final['yhat'] + final['xgb_residual']
        final[self.config.product_col] = pid
        # Fix 1: Clamp negatives — quantity sold can never be negative
        final['hybrid_yhat'] = final['hybrid_yhat'].clip(lower=0)
        final['yhat_lower']  = final['yhat_lower'].clip(lower=0)
        final['yhat_upper']  = final['yhat_upper'].clip(lower=0)
        # Fix 2+3: Only return fields backend needs (remove product_id, xgb_residual, raw yhat)
        return final[['ds', 'hybrid_yhat', 'yhat_lower', 'yhat_upper']]

    def build_forecasts(self, forecast_df: pd.DataFrame, horizons: Tuple[int, int]) -> Dict[str, pd.DataFrame]:
        outputs = {}
        for h in horizons:
            frames = []
            for pid, group in forecast_df.groupby(self.config.product_col):
                group = group.sort_values(self.config.date_col).reset_index(drop=True)
                if len(group) < self.config.min_history_points:
                    continue
                fc = self.forecast_horizon_for_product(group, h)
                if not fc.empty:
                    fc['horizon_days'] = h
                    frames.append(fc)
            outputs[f'{h}d'] = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        self.forecast_results = outputs
        return outputs

    def run(self, forecast_df: pd.DataFrame, prophet_df: pd.DataFrame) -> Dict[str, object]:
        forecast_df = self.prepare_base(forecast_df.copy())
        prophet_df = prophet_df.sort_values([self.config.product_col, 'ds']).reset_index(drop=True)
        train_df, test_df = self.split_train_test(forecast_df)
        self.fit_prophet_per_product(prophet_df)
        residuals_df = self.prophet_in_sample(prophet_df)
        self.train_xgb_residual_model(train_df, residuals_df)
        metrics = self.evaluate(test_df, residuals_df)
        forecasts = self.build_forecasts(forecast_df, self.config.horizons)
        return {'train_df': train_df, 'test_df': test_df, 'residuals_df': residuals_df, 'forecasts': forecasts, 'metrics': metrics}

## 8) Module 5 price optimization class
This class trains a log-log demand model and finds the price that maximizes revenue or profit.

In [8]:
class PriceOptimizationEngine:
    def __init__(self, config: Module5Config):
        self.config = config
        self.df: Optional[pd.DataFrame] = None
        self.feature_columns: List[str] = []
        self.metrics: Dict[str, float] = {}
        self.model = None
        self.best_reference_row = None
        self.label_encoders: Dict[str, LabelEncoder] = {}

    def load_data(self) -> pd.DataFrame:
        if not os.path.exists(self.config.pricing_csv_path):
            raise FileNotFoundError(f'Pricing dataset not found: {self.config.pricing_csv_path}')
        df = pd.read_csv(self.config.pricing_csv_path)
        if self.config.date_col in df.columns:
            df[self.config.date_col] = pd.to_datetime(df[self.config.date_col], errors='coerce')
        return df

    def prepare_base(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy().drop_duplicates()
        for col in [self.config.target_col, self.config.price_col, self.config.cost_col, self.config.sales_col, self.config.profit_col]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna(subset=[self.config.target_col, self.config.price_col]).reset_index(drop=True)
        df = df[df[self.config.target_col] > 0]
        df = df[df[self.config.price_col] > 0]
        return df

    def create_time_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        if self.config.date_col in df.columns:
            d = df[self.config.date_col]
            df['month'] = d.dt.month
            df['quarter'] = d.dt.quarter
            df['year'] = d.dt.year
            df['day_of_week'] = d.dt.dayofweek
            df['is_weekend'] = d.dt.dayofweek.isin([5, 6]).astype(int)
        return df

    def create_business_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        if self.config.cost_col in df.columns:
            df['unit_margin_inr'] = df[self.config.price_col] - df[self.config.cost_col]
            df['margin_pct'] = np.where(df[self.config.price_col] != 0, df['unit_margin_inr'] / df[self.config.price_col], 0)
        if self.config.sales_col in df.columns and self.config.target_col in df.columns:
            qty = df[self.config.target_col].replace(0, np.nan)
            df['realized_price'] = (df[self.config.sales_col] / qty).replace([np.inf, -np.inf], np.nan)
        if self.config.profit_col in df.columns:
            df['profit_flag'] = (df[self.config.profit_col] > 0).astype(int)
        df['log_price'] = np.log(df[self.config.price_col])
        df['log_quantity'] = np.log(df[self.config.target_col])
        return df

    def encode_categoricals(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        cat_cols = [c for c in df.select_dtypes(include=['object']).columns.tolist() if c not in [self.config.date_col]]
        self.label_encoders = {}
        for col in cat_cols:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            self.label_encoders[col] = le
        return df

    def choose_relevant_columns(self, df: pd.DataFrame) -> List[str]:
        exclude = {self.config.target_col, self.config.price_col, self.config.sales_col, self.config.profit_col, self.config.date_col}
        cols = [c for c in df.columns if c not in exclude]
        cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
        if 'log_price' in cols:
            cols = ['log_price'] + [c for c in cols if c != 'log_price']
        return cols

    def build_model_frame(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
        df = self.create_time_features(df)
        df = self.create_business_features(df)
        df = self.encode_categoricals(df)
        df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['log_price', 'log_quantity']).reset_index(drop=True)
        self.feature_columns = self.choose_relevant_columns(df)
        X = df[self.feature_columns].copy().fillna(0)
        y = df['log_quantity'].copy()
        return X, y

    def fit_model(self, X_train: pd.DataFrame, y_train: pd.Series):
        model = Ridge(alpha=1.0, random_state=self.config.random_state)
        model.fit(X_train, y_train)
        self.model = model
        return model

    def evaluate(self, X_test: pd.DataFrame, y_test: pd.Series) -> Dict[str, float]:
        y_pred_log = self.model.predict(X_test)
        y_true = np.exp(y_test.values)
        y_pred = np.exp(y_pred_log)
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        self.metrics = {'MAE': float(mae), 'RMSE': float(rmse), 'R2': float(r2)}
        return self.metrics

    def estimate_elasticity(self) -> float:
        if 'log_price' not in self.feature_columns or self.model is None:
            return np.nan
        idx = self.feature_columns.index('log_price')
        return float(self.model.coef_[idx])

    def simulate_curve(self, base_row: pd.Series) -> pd.DataFrame:
        current_price = float(base_row[self.config.price_col])
        pmin = current_price * self.config.candidate_price_range[0]
        pmax = current_price * self.config.candidate_price_range[1]
        price_grid = np.linspace(pmin, pmax, self.config.price_grid_points)

        rows = []
        for p in price_grid:
            row = base_row.copy()
            row[self.config.price_col] = p
            if self.config.cost_col in row.index and not pd.isna(row[self.config.cost_col]):
                row['unit_margin_inr'] = p - float(row[self.config.cost_col])
                row['margin_pct'] = row['unit_margin_inr'] / p if p != 0 else 0
            row['log_price'] = np.log(max(p, 1e-9))

            feat = pd.DataFrame([row]).reindex(columns=self.feature_columns, fill_value=0).fillna(0)
            pred_q = float(np.exp(self.model.predict(feat)[0]))
            revenue = p * pred_q
            profit = None
            if self.config.cost_col in row.index and not pd.isna(row[self.config.cost_col]):
                profit = (p - float(row[self.config.cost_col])) * pred_q
            rows.append({'price': float(p), 'predicted_quantity': float(pred_q), 'predicted_revenue': float(revenue), 'predicted_profit': float(profit) if profit is not None else None})

        return pd.DataFrame(rows)

    def optimize(self, df: pd.DataFrame) -> Dict[str, object]:
        base_row = df.iloc[0].copy()
        self.best_reference_row = base_row
        curve = self.simulate_curve(base_row)
        best_rev_row = curve.loc[curve['predicted_revenue'].idxmax()].to_dict()
        best_profit_row = None
        if curve['predicted_profit'].notna().any():
            best_profit_row = curve.loc[curve['predicted_profit'].idxmax()].to_dict()
        return {'current_price': float(base_row[self.config.price_col]), 'elasticity': self.estimate_elasticity(), 'curve': curve, 'best_revenue': best_rev_row, 'best_profit': best_profit_row}

    def run(self) -> Dict[str, object]:
        df = self.load_data()
        df = self.prepare_base(df)
        if self.config.product_col in df.columns:
            sort_cols = [self.config.product_col] + ([self.config.date_col] if self.config.date_col in df.columns else [])
            df = df.sort_values(sort_cols).reset_index(drop=True)
        X, y = self.build_model_frame(df)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=self.config.test_size, random_state=self.config.random_state)
        self.fit_model(X_train, y_train)
        metrics = self.evaluate(X_test, y_test)
        optimization = self.optimize(df)

        result = {
            'metrics': metrics,
            'elasticity': optimization['elasticity'],
            'current_price': optimization['current_price'],
            'best_revenue': optimization['best_revenue'],
            'best_profit': optimization['best_profit'],
            'curve': optimization['curve'],
            'sample_rows': df.head(5),
        }

        os.makedirs(self.config.output_dir, exist_ok=True)
        optimization['curve'].to_csv(os.path.join(self.config.output_dir, 'module5_price_curve.csv'), index=False)
        pd.DataFrame([optimization['best_revenue']]).to_csv(os.path.join(self.config.output_dir, 'module5_best_revenue.csv'), index=False)
        if optimization['best_profit'] is not None:
            pd.DataFrame([optimization['best_profit']]).to_csv(os.path.join(self.config.output_dir, 'module5_best_profit.csv'), index=False)
        with open(os.path.join(self.config.output_dir, 'module5_metrics.json'), 'w') as f:
            json.dump({k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in metrics.items()}, f, indent=2)

        return result

## 9) Run Module 4 and Module 5
Module 4 uses the forecasting dataset from Module 3. Module 5 uses the pricing dataset from Module 3.

In [9]:
# Run Module 3 first
m3_processor = Module3Preprocessor(config3)
m3 = m3_processor.run()
display(m3['summary_df'])

# Run Module 4 using Module 3 outputs
m4_processor = HybridDemandForecaster(config4)
m4 = m4_processor.run(m3['forecast_df'], m3['prophet_df'])
print('Module 4 Evaluation Metrics:')
print(m4['metrics'])

# Run Module 5 using Module 3 pricing dataset
m5_engine = PriceOptimizationEngine(config5)
m5 = m5_engine.run()
print('Module 5 Evaluation Metrics:')
print(m5['metrics'])
print('Elasticity:', m5['elasticity'])
print('Current Price:', m5['current_price'])
print('Best Revenue Price:', m5['best_revenue'])
print('Best Profit Price:', m5['best_profit'])

,dataset,rows,columns
0,raw_input,10100,28
1,cleaned_transaction_data,9812,28
2,daily_product_aggregated_data,3387,17
3,forecasting_dataset,2868,52
4,pricing_dataset,9812,44
5,anomaly_dataset,9812,44


INFO:prophet:n_changepoints greater than number of observations. Using 15.
INFO:prophet:n_changepoints greater than number of observations. Using 21.
INFO:prophet:n_changepoints greater than number of observations. Using 15.
INFO:prophet:n_changepoints greater than number of observations. Using 15.
INFO:prophet:n_changepoints greater than number of observations. Using 17.
INFO:prophet:n_changepoints greater than number of observations. Using 18.


Module 4 Evaluation Metrics:
{'MAE': 14.296401559601634, 'RMSE': 24.954745532825747, 'MAPE': 188.0389032970795}
Module 5 Evaluation Metrics:
{'MAE': 0.007758075640687158, 'RMSE': 0.0811305038114543, 'R2': 0.9999869368749025}
Elasticity: -0.005175544158382555
Current Price: 0.0015127770307444
Best Revenue Price: {'price': 0.00196661013996772, 'predicted_quantity': 1.0616694986713773, 'predicted_revenue': 0.002087890001381576, 'predicted_profit': 0.00025105919918554104}
Best Profit Price: {'price': 0.00196661013996772, 'predicted_quantity': 1.0616694986713773, 'predicted_revenue': 0.002087890001381576, 'predicted_profit': 0.00025105919918554104}


## 10) Save JSON responses
These JSON files can be used by a backend or frontend application.

In [10]:
def json_safe(value):
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return value


# ── Fix 6: Anomaly detection (new task output) ───────────────────────────────
def detect_anomalies_response(df, date_col='order_date', qty_col='quantity_sold', recent_days=7, threshold=2.5):
    """Z-score anomaly detection returning backend-compatible JSON."""
    df = df.copy().sort_values(date_col).reset_index(drop=True)
    qty = df[qty_col].astype(float)
    mean_qty = float(qty.mean())
    std_qty  = float(qty.std()) if qty.std() > 0 else 1.0
    last_date = df[date_col].max()
    cutoff = last_date - pd.Timedelta(days=recent_days)
    flagged = []
    for _, row in df.iterrows():
        q = float(row[qty_col])
        z = (q - mean_qty) / std_qty
        if abs(z) < threshold:
            continue
        atype = 'spike' if z > 0 else 'drop'
        stage = 'post_upload_alert' if row[date_col] >= cutoff else 'pre_forecast_historical'
        direction = 'above' if atype == 'spike' else 'below'
        explanation = (
            f"Sales quantity ({int(q)}) is {round(min(abs(z), 10), 2)} standard deviations "
            f"{direction} the historical mean ({round(mean_qty, 1)}). "
            f"{'Possible promotion or demand surge.' if atype == 'spike' else 'Possible supply disruption or stockout.'}"
        )
        flagged.append({
            'date': row[date_col].isoformat() if hasattr(row[date_col], 'isoformat') else str(row[date_col]),
            'stage': stage,
            'anomaly_type': atype,
            'severity_score': round(min(abs(z), 10.0), 2),
            'explanation': explanation,
            'acknowledged': False,
        })
    return {'model_version': '1.0.0', 'flagged_anomalies': flagged}


# ── Build Forecasting Response (Fix 4+5) ─────────────────────────────────────
forecast_7d_rows  = m4['forecasts'].get('7d',  pd.DataFrame()).copy()
forecast_30d_rows = m4['forecasts'].get('30d', pd.DataFrame()).copy()

# Add horizon_days label (Fix 4: horizon_days was already in build_forecasts)
if not forecast_7d_rows.empty and 'horizon_days' not in forecast_7d_rows.columns:
    forecast_7d_rows['horizon_days'] = 7
if not forecast_30d_rows.empty and 'horizon_days' not in forecast_30d_rows.columns:
    forecast_30d_rows['horizon_days'] = 30

module4_response = {
    'metrics':      m4['metrics'],                                       # MAE, RMSE, MAPE, sMAPE
    'forecast_7d':  forecast_7d_rows.to_dict(orient='records'),          # 7-day forecasts
    'forecast_30d': forecast_30d_rows.to_dict(orient='records'),         # 30-day forecasts
    # Fix 5: removed 'summary', 'm3_shape', 'm4_shape' — not part of backend contract
}


# ── Build Pricing Response (Fix 6: correct backend schema) ───────────────────
current_p = float(m5['current_price'])
bound_pct = 0.20
bound_min = round(current_p * (1 - bound_pct), 2)
bound_max = round(current_p * (1 + bound_pct), 2)
best_rev  = m5['best_revenue']
curve_df  = m5['curve']

# Pick 5 evenly-spaced candidates from the simulated price curve
n = len(curve_df)
indices = [0, n // 4, n // 2, 3 * n // 4, n - 1]
candidate_grid = [
    {
        'candidate_price':   round(float(curve_df.iloc[i]['price']), 2),
        'estimated_demand':  round(float(curve_df.iloc[i]['predicted_quantity']), 2),
        'estimated_revenue': round(float(curve_df.iloc[i]['predicted_revenue']), 2),
    }
    for i in indices
]

import math
elasticity = m5.get('elasticity')
has_price_variation = elasticity is not None and not (isinstance(elasticity, float) and math.isnan(elasticity))

module5_response = {
    'eligibility_status':    'eligible' if has_price_variation else 'insufficient_price_variation',
    'eligibility_reason':    None if has_price_variation else 'No price variation detected in history. Elasticity cannot be estimated.',
    'recommended_price':     round(float(best_rev['price']), 2) if has_price_variation else None,
    'expected_revenue':      round(float(best_rev['predicted_revenue']), 2) if has_price_variation else None,
    'elasticity_model_type': 'Log-Log Ridge Elasticity',
    'model_version':         '1.0.0',
    'bound_range':           {'min': bound_min, 'max': bound_max} if has_price_variation else None,
    'candidate_grid':        candidate_grid if has_price_variation else None,
}


# ── Build Anomaly Response (Fix 6: new task) ──────────────────────────────────
anomaly_response = detect_anomalies_response(
    m3['cleaned_df'],
    date_col='order_date',
    qty_col='quantity_sold'
)


# ── Serialize and save ────────────────────────────────────────────────────────
safe_module4 = json.loads(json.dumps(module4_response, default=json_safe))
safe_module5 = json.loads(json.dumps(module5_response, default=json_safe))
safe_anomaly = json.loads(json.dumps(anomaly_response, default=json_safe))

os.makedirs('/content/drive/MyDrive/module4_outputs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/module5_outputs', exist_ok=True)

with open('/content/drive/MyDrive/module4_outputs/module4_response.json', 'w') as f:
    json.dump(safe_module4, f, indent=2)
with open('/content/drive/MyDrive/module5_outputs/module5_response.json', 'w') as f:
    json.dump(safe_module5, f, indent=2)
with open('/content/drive/MyDrive/module5_outputs/anomaly_response.json', 'w') as f:
    json.dump(safe_anomaly, f, indent=2)


# ── Validation summary ────────────────────────────────────────────────────────
print('=== FORECASTING ===')
print('Keys:', list(safe_module4.keys()))
print('forecast_7d rows :', len(safe_module4['forecast_7d']))
print('forecast_30d rows:', len(safe_module4['forecast_30d']))
neg7  = sum(1 for r in safe_module4['forecast_7d']  if r.get('hybrid_yhat', 0) < 0)
neg30 = sum(1 for r in safe_module4['forecast_30d'] if r.get('hybrid_yhat', 0) < 0)
print(f'Negative hybrid_yhat: {neg7} (7d),  {neg30} (30d)  — both should be 0')
extra = [k for k in ['product_id', 'xgb_residual', 'yhat', 'summary', 'm3_shape', 'm4_shape']
         if k in (safe_module4.get('forecast_7d', [{}])[0] if safe_module4.get('forecast_7d') else {}) or k in safe_module4]
print('Extra keys present (should be empty):', extra)

print()
print('=== PRICING ===')
print('Keys:', list(safe_module5.keys()))
print('eligibility_status:', safe_module5.get('eligibility_status'))
print('recommended_price :', safe_module5.get('recommended_price'))
print('candidate_grid items:', len(safe_module5.get('candidate_grid') or []))

print()
print('=== ANOMALY ===')
print('Keys:', list(safe_anomaly.keys()))
print('flagged_anomalies count:', len(safe_anomaly.get('flagged_anomalies', [])))

print()
print('All 3 JSON responses saved to Drive.')


Saved JSON responses to Drive


## 11) Preview outputs
Inspect the forecast and pricing outputs to confirm everything is working correctly.

In [11]:
display(m4['forecasts'].get('7d', pd.DataFrame()).head())
display(m4['forecasts'].get('30d', pd.DataFrame()).head())
display(m5['curve'].head())
display(m5['curve'].tail())

,product_id,ds,yhat,xgb_residual,hybrid_yhat,yhat_lower,yhat_upper,horizon_days
0,P1025,2024-12-28,-26.270312,-0.050408,-26.320720,-62.024693,11.731360,7
1,P1025,2024-12-29,-12.300850,-0.050408,-12.351258,-48.437917,24.379408,7
2,P1025,2024-12-30,2.818572,-0.044283,2.774289,-34.352538,38.732374,7
3,P1025,2024-12-31,-9.281765,-0.044460,-9.326225,-42.561707,27.841620,7
4,P1025,2025-01-01,385.816753,-0.044283,385.772470,346.956043,421.745848,7


,product_id,ds,yhat,xgb_residual,hybrid_yhat,yhat_lower,yhat_upper,horizon_days
0,P1025,2024-12-28,-26.270312,-0.050408,-26.320720,-64.071782,7.858332,30
1,P1025,2024-12-29,-12.300850,-0.050408,-12.351258,-49.323467,24.951447,30
2,P1025,2024-12-30,2.818572,-0.044283,2.774289,-35.601097,38.872493,30
3,P1025,2024-12-31,-9.281765,-0.044460,-9.326225,-44.250214,26.592690,30
4,P1025,2025-01-01,385.816753,-0.044283,385.772470,349.430662,419.714065,30


,price,predicted_quantity,predicted_revenue,predicted_profit
0,0.001059,1.065419,0.001128,-0.000715
1,0.001082,1.065283,0.001153,-0.000690
2,0.001105,1.065151,0.001178,-0.000665
3,0.001129,1.065021,0.001202,-0.000640
4,0.001152,1.064894,0.001227,-0.000616


,price,predicted_quantity,predicted_revenue,predicted_profit
35,0.001874,1.061956,0.001990,0.000152
36,0.001897,1.061883,0.002014,0.000177
37,0.001920,1.061811,0.002039,0.000202
38,0.001943,1.061740,0.002063,0.000226
39,0.001967,1.061669,0.002088,0.000251


## 12) Show saved file paths
These are the key output files generated by the notebook.

In [12]:
for name, path in {
    'module3_cleaned_transactions': '/content/drive/MyDrive/module3_outputs/module3_cleaned_transactions.csv',
    'module3_forecasting_dataset': '/content/drive/MyDrive/module3_outputs/module3_forecasting_dataset.csv',
    'module3_prophet_dataset': '/content/drive/MyDrive/module3_outputs/module3_prophet_dataset.csv',
    'module3_pricing_dataset': '/content/drive/MyDrive/module3_outputs/module3_pricing_dataset.csv',
    'module4_response_json': '/content/drive/MyDrive/module4_outputs/module4_response.json',
    'module5_price_curve_csv': '/content/drive/MyDrive/module5_outputs/module5_price_curve.csv',
    'module5_response_json': '/content/drive/MyDrive/module5_outputs/module5_response.json',
}.items():
    print(f'{name}: {path}')

module3_cleaned_transactions: /content/drive/MyDrive/module3_outputs/module3_cleaned_transactions.csv
module3_forecasting_dataset: /content/drive/MyDrive/module3_outputs/module3_forecasting_dataset.csv
module3_prophet_dataset: /content/drive/MyDrive/module3_outputs/module3_prophet_dataset.csv
module3_pricing_dataset: /content/drive/MyDrive/module3_outputs/module3_pricing_dataset.csv
module4_response_json: /content/drive/MyDrive/module4_outputs/module4_response.json
module5_price_curve_csv: /content/drive/MyDrive/module5_outputs/module5_price_curve.csv
module5_response_json: /content/drive/MyDrive/module5_outputs/module5_response.json
